# XM655 two sided full duplex

Both nodes transmit and receive at the same time, and each one cancels its own
signal out of its own receivers - on the board, not on paper.

- **BSIC** beamforms around the interference: transmit weights that spill little
  on our own receivers, receive weights that hear little of what is left.
- **DSIC** fits the known transmit waveform to what BSIC left and subtracts it,
  after the ADC - so a saturated front end stays saturated.

Each side transmits a **different tone**. That is what makes the measurement
readable: at one FFT bin sits the interference a side is trying to kill, at
another sits the link it must not damage, and the two never mix.

The stored channels only **design** BSIC. Every reported number is **measured**,
over five aligned captures, so estimation error shows up instead of hiding.

Joint BSIC lives in `lib/fd/joint_sic_bsic.py`, the non joint fallback in
`lib/fd/non_joint_sic_bsic.py`, DSIC in `lib/fd/dsic.py`.

## 1. Parameters

Everything that gets tuned. Nothing outside this cell should need editing.

The RF block and the four system lists must match the estimation run - section 2
checks them against the stored `params.json`.

Both sides are live from the first capture to the last. Side 1's tone is set here
and not in the shared config, because it is the only notebook that has two.

In [ ]:
from lib.config_parser import load_config

CFG = load_config()

# --- side 0 ---
DACS_0 = CFG["systems"]["dacs_0"]    # overlay.dac[] indices, tile 2
ADCS_0 = CFG["systems"]["adcs_0"]    # its own receivers, tile 1

# --- side 1 ---
DACS_1 = CFG["systems"]["dacs_1"]    # overlay.dac[] indices, tile 3
ADCS_1 = CFG["systems"]["adcs_1"]    # its own receivers, tile 0

# --- the estimated channels bsic is designed on ---
CHANNELS_DIR = "output/channels"     # where channel_estimation.ipynb left them

# --- rf: must match the estimation run, checked below ---
DAC_NCO  = CFG["rf"]["dac_nco"]      # MHz, per tile -> TX lands at 4900
DAC_ZONE = CFG["rf"]["dac_zone"]     # Nyquist zone, per tile
ADC_NCO  = CFG["rf"]["adc_nco"]      # MHz, per tile = 5000 - DAC_NCO
ADC_ZONE = CFG["rf"]["adc_zone"]     # the fold is in an even zone

# --- rates: fixed by the bitstream, do not change ---
DAC_SR = CFG["board"]["dac_sr"]      # DAC baseband rate = 10 GSPS / 10 (C2R eats one x2)
ADC_SR = CFG["board"]["adc_sr"]      # ADC baseband rate = 2.5 GSPS / 10 decimation
N_CH   = CFG["board"]["n_ch"]        # RF channels on the XM655
N_TILE = CFG["board"]["n_tile"]      # ADC and DAC tiles

PATH_PER_TILE = N_CH // N_TILE       # converters per tile - one player memory each

# --- one tone per side, so the FFT can tell them apart ---
CW_TONE_0_MHZ = CFG["signal"]["tone_0_mhz"]  # side 0 transmits this
CW_TONE_1_MHZ = CFG["signal"]["tone_1_mhz"]  # side 1 transmits this
CW_AMP        = CFG["signal"]["amp"]       # 14 bit DAC: +16383 / -16384

# --- bsic: which beamforming canceller, and what it needs ---
BSIC_ALGORITHM = "non_joint_max_sinr"   # joint_* live in lib/fd/joint_sic_bsic.py
SNR_DB        = 40         # the transmission against the board's own noise floor
BSIC_MAX_GAIN = 1.0        # ceiling on any element gain, the register allows 2.0
MAX_PHASE_DEG = 179.99     # the converter rejects exactly 180, so clamp to this

# --- dsic: which digital canceller, and what it needs ---
DSIC_ALGORITHM = "linear_lms"           # see lib/fd/dsic.py
DSIC_TAPS    = 1           # delays the fit is allowed, per side
DSIC_MU      = 1e-3        # lms step: larger converges faster and settles noisier
DSIC_TRAIN_N = 16384       # preamble length: DSIC_MU * this is how far it settles

# --- one block of samples ---
N_CAP       = CFG["capture"]["n_cap"]        # samples per channel
TRIG_HOLD_S = CFG["capture"]["trig_hold_s"]  # trig_cap must stay high for a whole capture window

# --- where this run is written ---
SIC_DIR = "output/two_side_sic"

## 2. Verify the parameters

Catch a bad settings cell before anything touches the board, and confirm the
stored matrices were measured on *this* array at *this* frequency.

Two checks are new here and worth knowing about:

- **One tile per side.** There is one player memory per DAC tile, so a side that
  transmits its own tone has to sit inside one tile. Split a side across two and
  both would play whatever was written last.
- **Different FFT bins.** Both tones are snapped onto the capture grid, then
  checked to land on bins of their own. Sharing a bin would make the interference
  and the link the same number.

The channels were measured at side 0's tone. Side 1 transmits a little away from
it, so its beamformer assumes the array looks the same over that offset - true for
half a megahertz on an array this size, and less true the wider the split gets.

In [ ]:
import json
import os

import numpy as np

from lib.fd.non_joint_sic_bsic import BSIC_ALGORITHMS
from lib.fd.joint_sic_bsic import JOINT_BSIC_ALGORITHMS
from lib.fd.dsic import DSIC_ALGORITHMS
from lib.common_functions import capture_aligned
from lib.common_functions import clear_dir
from lib.common_functions import convert_raw_to_iq
from lib.common_functions import create_tone_samples
from lib.common_functions import find_tone_bin
from lib.common_functions import save_json_params
from lib.common_functions import snap_tone_to_fft_bin
from lib.common_functions import tune_adcs
from lib.common_functions import tune_dacs
from lib.common_functions import write_tone_to_tile_player

ALL_BSIC_ALGORITHMS = dict(BSIC_ALGORITHMS)
ALL_BSIC_ALGORITHMS.update(JOINT_BSIC_ALGORITHMS)

def validate_systems():
    """Both sides are complete and disjoint, and each transmits from one DAC tile."""
    for name, indices in (("DACS_0", DACS_0), ("ADCS_0", ADCS_0),
                          ("DACS_1", DACS_1), ("ADCS_1", ADCS_1)):
        if len(indices) == 0:
            raise ValueError("%s is empty - both sides transmit and receive here" % name)
        if not set(indices) <= set(range(N_CH)):
            raise ValueError("%s holds indices outside 0..%d" % (name, N_CH - 1))
        if len(set(indices)) != len(indices):
            raise ValueError("%s lists the same index twice" % name)
    shared_dacs = set(DACS_0) & set(DACS_1)
    if shared_dacs:
        raise ValueError("DACs %s are listed under both sides" % sorted(shared_dacs))
    shared_adcs = set(ADCS_0) & set(ADCS_1)
    if shared_adcs:
        raise ValueError("ADCs %s are listed under both sides" % sorted(shared_adcs))
    tile_of = {}
    for name, dacs in (("DACS_0", DACS_0), ("DACS_1", DACS_1)):
        tiles = sorted(set(dac // PATH_PER_TILE for dac in dacs))
        if len(tiles) != 1:
            raise ValueError("%s spans DAC tiles %s - there is one player memory per "
                             "tile, so a side sending its own tone must sit inside one"
                             % (name, tiles))
        tile_of[name] = tiles[0]
    if tile_of["DACS_0"] == tile_of["DACS_1"]:
        raise ValueError("both sides transmit from DAC tile %d - they would share one "
                         "player memory and could not send different tones"
                         % tile_of["DACS_0"])

def validate_rf_settings():
    """Converter tables are per tile, the folds add up, and both tones fit the band."""
    for name, table in (("DAC_NCO", DAC_NCO), ("DAC_ZONE", DAC_ZONE),
                        ("ADC_NCO", ADC_NCO), ("ADC_ZONE", ADC_ZONE)):
        if len(table) != N_TILE:
            raise ValueError("%s needs one entry per tile (%d), got %d"
                             % (name, N_TILE, len(table)))
    fold_mhz = 2 * (10 * ADC_SR / 1e6)
    for tile, (dac_nco, adc_nco) in enumerate(zip(DAC_NCO, ADC_NCO)):
        if abs(dac_nco + adc_nco - fold_mhz) > 1e-6:
            raise ValueError("tile %d: ADC_NCO should be %g - DAC_NCO"
                             % (tile, fold_mhz))
    for name, tone_mhz in (("CW_TONE_0_MHZ", CW_TONE_0_MHZ),
                           ("CW_TONE_1_MHZ", CW_TONE_1_MHZ)):
        if not 0 < tone_mhz < ADC_SR / 2e6:
            raise ValueError("%s must be between 0 and %g, got %g"
                             % (name, ADC_SR / 2e6, tone_mhz))
    if not 0 < CW_AMP <= 16383:
        raise ValueError("CW_AMP must be between 1 and 16383")
    if N_CAP <= 0 or N_CAP & (N_CAP - 1):
        raise ValueError("N_CAP must be a positive power of two, got %d" % N_CAP)
    if TRIG_HOLD_S <= 0:
        raise ValueError("TRIG_HOLD_S must be positive")

def validate_tone_separation():
    """The two tones sit on bins of their own, or neither can be read alone."""
    bin_0 = find_tone_bin(CW_TONE_0_MHZ, N_CAP, ADC_SR)
    bin_1 = find_tone_bin(CW_TONE_1_MHZ, N_CAP, ADC_SR)
    if bin_0 == bin_1:
        raise ValueError("both tones land on FFT bin %d - they must differ by at least "
                         "one bin of the %g Hz grid" % (bin_0, ADC_SR / N_CAP))
    print("side 0: %.6f MHz on bin %d | side 1: %.6f MHz on bin %d | %d bins apart"
          % (CW_TONE_0_MHZ, bin_0, CW_TONE_1_MHZ, bin_1, abs(bin_1 - bin_0)))

def validate_algorithms():
    """Both algorithm names resolve, their settings are sane, and say what they buy."""
    if BSIC_ALGORITHM not in ALL_BSIC_ALGORITHMS:
        raise ValueError("%r is not a BSIC algorithm. Valid names: %s"
                         % (BSIC_ALGORITHM, ", ".join(ALL_BSIC_ALGORITHMS)))
    if DSIC_ALGORITHM not in DSIC_ALGORITHMS:
        raise ValueError("%r is not a DSIC algorithm. Valid names: %s"
                         % (DSIC_ALGORITHM, ", ".join(DSIC_ALGORITHMS)))
    if SNR_DB <= 0:
        raise ValueError("SNR_DB must be positive, got %g" % SNR_DB)
    if not 0 < BSIC_MAX_GAIN <= 2.0:
        raise ValueError("BSIC_MAX_GAIN must be between 0 and 2.0, got %g"
                         % BSIC_MAX_GAIN)
    if not 90.0 <= MAX_PHASE_DEG < 180.0:
        raise ValueError("MAX_PHASE_DEG must be just under 180, got %g" % MAX_PHASE_DEG)
    if DSIC_TAPS < 1:
        raise ValueError("DSIC_TAPS must be at least 1, got %d" % DSIC_TAPS)
    if DSIC_MU <= 0:
        raise ValueError("DSIC_MU must be positive, got %g" % DSIC_MU)
    if DSIC_TRAIN_N < DSIC_TAPS:
        raise ValueError("DSIC_TRAIN_N %d is shorter than DSIC_TAPS %d"
                         % (DSIC_TRAIN_N, DSIC_TAPS))
    if DSIC_TRAIN_N > N_CAP:
        raise ValueError("DSIC_TRAIN_N %d is longer than the block, N_CAP is %d"
                         % (DSIC_TRAIN_N, N_CAP))
    settling_db = -20 * np.log10(np.exp(-DSIC_MU * DSIC_TRAIN_N))
    joint = "joint" if BSIC_ALGORITHM in JOINT_BSIC_ALGORITHMS else "per side"
    print("bsic: %s, designed %s, %d element(s) each, for %g dB SNR"
          % (BSIC_ALGORITHM, joint, len(DACS_0), SNR_DB))
    print("dsic: %s, %d tap(s), step %g over %d sample(s) - the taps walk to "
          "within %.0f dB of the channel"
          % (DSIC_ALGORITHM, DSIC_TAPS, DSIC_MU, DSIC_TRAIN_N, settling_db))

def validate_stored_channels():
    """All four matrices are there, the right shape, and from this exact setup."""
    if not os.path.isdir(CHANNELS_DIR):
        raise FileNotFoundError("%s does not exist - run channel_estimation.ipynb to "
                                "measure the channels first" % CHANNELS_DIR)
    for name in ("h00.npy", "h01.npy", "h10.npy", "h11.npy", "params.json"):
        path = os.path.join(CHANNELS_DIR, name)
        if not os.path.isfile(path):
            raise FileNotFoundError("%s is missing - run channel_estimation.ipynb to "
                                    "measure the channels first" % path)
    wanted = {"h00.npy": (len(ADCS_0), len(DACS_0)),
              "h01.npy": (len(ADCS_0), len(DACS_1)),
              "h10.npy": (len(ADCS_1), len(DACS_0)),
              "h11.npy": (len(ADCS_1), len(DACS_1))}
    for name, expected in wanted.items():
        shape = np.load(os.path.join(CHANNELS_DIR, name)).shape
        if shape != expected:
            raise ValueError("%s has shape %s, this array wants %s"
                             % (name, shape, expected))
    with open(os.path.join(CHANNELS_DIR, "params.json")) as handle:
        stored = json.load(handle)
    for key, mine in (("dacs_0", DACS_0), ("adcs_0", ADCS_0),
                      ("dacs_1", DACS_1), ("adcs_1", ADCS_1),
                      ("dac_nco", DAC_NCO), ("adc_nco", ADC_NCO)):
        if list(stored[key]) != list(mine):
            raise ValueError("%s: the stored channels say %s, this notebook says %s"
                             % (key, stored[key], mine))
    if abs(stored["cw_tone_mhz"] - CW_TONE_0_MHZ) > 1e-9:
        raise ValueError("the stored channels were measured at %g MHz, not %g MHz"
                         % (stored["cw_tone_mhz"], CW_TONE_0_MHZ))
    print("channels from %s, measured at %.6f MHz - side 1 transmits %.6f MHz away"
          % (CHANNELS_DIR, stored["cw_tone_mhz"],
             abs(CW_TONE_1_MHZ - stored["cw_tone_mhz"])))

validate_systems()
validate_rf_settings()
CW_TONE_0_MHZ = snap_tone_to_fft_bin(CW_TONE_0_MHZ, ADC_SR, N_CAP)
CW_TONE_1_MHZ = snap_tone_to_fft_bin(CW_TONE_1_MHZ, ADC_SR, N_CAP)
validate_tone_separation()
validate_algorithms()
validate_stored_channels()

print("parameters ok - side 0: DAC %s ADC %s | side 1: DAC %s ADC %s"
      % (DACS_0, ADCS_0, DACS_1, ADCS_1))
print("algorithms available - bsic: %s | dsic: %s"
      % (", ".join(ALL_BSIC_ALGORITHMS), ", ".join(DSIC_ALGORITHMS)))

## 3. Load the estimated channels

All four matrices are used here. That is the difference from the one sided bench:
side 1 is a transmitter now, so its own leakage `H11` matters as much as ours.

| key | path | shape |
|---|---|---|
| `h00` | side 0 DACs -> side 0 ADCs, side 0's interference | `(len(ADCS_0), len(DACS_0))` |
| `h01` | side 1 DACs -> side 0 ADCs, side 0's link | `(len(ADCS_0), len(DACS_1))` |
| `h10` | side 0 DACs -> side 1 ADCs, side 1's link | `(len(ADCS_1), len(DACS_0))` |
| `h11` | side 1 DACs -> side 1 ADCs, side 1's interference | `(len(ADCS_1), len(DACS_1))` |

The eigenvalue spread printed per side is **the ceiling on what BSIC can do**
there: all of them within a few dB means nowhere quiet to stand, and no transmit
weight helps.

In [ ]:
def load_estimated_channels():
    """All four matrices - a two sided link needs every one of them."""
    channels = {}
    for name in ("h00", "h01", "h10", "h11"):
        channels[name] = np.load(os.path.join(CHANNELS_DIR, name + ".npy"))
    return channels

def describe_channels(channels):
    """One line per matrix, then each side's headroom against its own leakage."""
    for name in ("h00", "h01", "h10", "h11"):
        matrix = channels[name]
        print("%-5s %s  |h| min %.4g  max %.4g"
              % (name, matrix.shape, np.abs(matrix).min(), np.abs(matrix).max()))
    for side, name in ((0, "h00"), (1, "h11")):
        si = channels[name]
        eigenvalues = np.maximum(np.linalg.eigvalsh(si.conj().T @ si), 0.0)
        quietest = max(eigenvalues[0], eigenvalues[-1] * 1e-12)
        print("side %d %s eigenvalues: %s"
              % (side, name.upper(),
                 np.array2string(eigenvalues, precision=4, suppress_small=True)))
        print("side %d bsic headroom: about %.1f dB between its quietest direction "
              "and its loudest" % (side, 10 * np.log10(eigenvalues[-1] / quietest)))

CHANNELS = load_estimated_channels()
describe_channels(CHANNELS)

## 4. BSIC - beamforming cancellation

`BSIC_ALGORITHM` picks the beamformer. Two families are on offer:

| family | file | what it gets |
|---|---|---|
| `joint_*` | `lib/fd/joint_sic_bsic.py` | all four matrices at once, both sides solved together |
| `non_joint_*` | `lib/fd/non_joint_sic_bsic.py` | run once per side, each side blind to what the other chose |

The joint pair alternates all four weights until they agree: each side's transmit
weight solved against the beam the other side is receiving with, each receive
weight against the beam the other side is transmitting with, twenty rounds from a
uniform start. `joint_max_sinr` trades its own leakage against the noise floor,
`joint_zf` nulls it outright and spends one receive element doing so.

The non joint three are the baseline they have to beat: side 0 designed on
`(H00, H10, H01)`, side 1 on `(H11, H01, H10)`, neither knowing the other's answer.
`non_joint_max_sinr` trades leakage against the noise floor over the whole
interference subspace, `non_joint_zf` nulls it at the receiver, and
`non_joint_softnull_max_sinr_based` transmits down the quietest direction of its own
`H00` first, so the leakage is already small at the antenna and the receive weight
only has to beat what is left of it.

Four vectors come out, and only two of them are hardware:

| | weights | where it lives |
|---|---|---|
| `tx_0`, `tx_1` | `DACS_0`, `DACS_1` | the gain and phase registers, one per DAC |
| `rx_0`, `rx_1` | `ADCS_0`, `ADCS_1` | a complex sum in numpy, straight after the capture |

Each side is scaled to `BSIC_MAX_GAIN` on its **own** peak element, so neither
side's ceiling squashes the other.

In [ ]:
def design_bsic(channels):
    """Weights for both sides, from a joint algorithm or the non joint one run twice."""
    noise_power = 10 ** (-SNR_DB / 10)
    if BSIC_ALGORITHM in JOINT_BSIC_ALGORITHMS:
        algorithm = JOINT_BSIC_ALGORITHMS[BSIC_ALGORITHM]
        return algorithm(channels["h00"], channels["h01"], channels["h10"],
                         channels["h11"], noise_power)
    algorithm = BSIC_ALGORITHMS[BSIC_ALGORITHM]
    side_0 = algorithm(channels["h00"], channels["h10"], channels["h01"], noise_power)
    side_1 = algorithm(channels["h11"], channels["h01"], channels["h10"], noise_power)
    return {"tx_0": side_0["tx"], "rx_0": side_0["nf_rx"],
            "tx_1": side_1["tx"], "rx_1": side_1["nf_rx"]}

def create_uniform_weights(channels):
    """Every element driven and heard equally - the reference BSIC has to beat."""
    weights = {}
    for side, name in ((0, "h00"), (1, "h11")):
        receivers, elements = channels[name].shape
        weights["tx_%d" % side] = np.ones(elements, dtype=complex) / np.sqrt(elements)
        weights["rx_%d" % side] = np.ones(receivers, dtype=complex) / np.sqrt(receivers)
    return weights

def create_hardware_weights(weights):
    """Both transmit vectors as the gain and phase tables the DACs take.

    The ceiling scales a side as a whole and not each element, because the beam
    lives in the ratios between them.
    """
    gains = [0.0] * N_CH
    phases = [0.0] * N_CH
    for side, dacs in ((0, DACS_0), (1, DACS_1)):
        vector = weights["tx_%d" % side]
        peak = np.max(np.abs(vector))
        for element, dac in enumerate(dacs):
            weight = vector[element] * BSIC_MAX_GAIN / peak
            degrees = (np.rad2deg(np.angle(weight)) + 180.0) % 360.0 - 180.0
            gains[dac] = abs(weight)
            phases[dac] = max(-MAX_PHASE_DEG, min(MAX_PHASE_DEG, degrees))
    return gains, phases

def create_silent_gains(gains, dacs):
    """The same gain table with one side muted, for a capture it must not be in."""
    silent = list(gains)
    for dac in dacs:
        silent[dac] = 0.0
    return silent

BSIC_WEIGHTS = design_bsic(CHANNELS)
BSIC_GAINS, BSIC_PHASES = create_hardware_weights(BSIC_WEIGHTS)

for side, dacs in ((0, DACS_0), (1, DACS_1)):
    for dac in dacs:
        print("side %d DAC %2d: gain %.4f, phase %+8.3f deg"
              % (side, dac, BSIC_GAINS[dac], BSIC_PHASES[dac]))
    print("side %d rx combine: %s"
          % (side, np.round(BSIC_WEIGHTS["rx_%d" % side], 4)))

## 5. Set up the board

The same state the channels were measured in - weights designed on them only mean
something if the board is put back into it.

Each side's tone goes into **its own tile's player memory** once and is never
rewritten. All four DACs of a tile read that player, so the gain and phase table
alone decides how a side's four elements combine, and every capture transmits the
same two waveforms with the same phase - the only reason a canceller fitted on one
block can run on the next. Muting a side for a training capture is a gain of zero
on its DACs, not a rewritten player.

`capture_aligned()` fires the one trigger edge that starts both transmitters and
all the receivers together. `get_custom_data_xm655()` is not used: it fires its own
trigger and would undo the alignment.

In [ ]:
from lib.mts import doaMtsOverlay

def setup_board():
    """Load the overlay, tune both converter sets, and give each side its own tone."""
    overlay = doaMtsOverlay("mts.bit")
    tune_dacs(overlay, DAC_NCO, DAC_ZONE, N_CH)
    tune_adcs(overlay, ADC_NCO, ADC_ZONE, N_CH, N_CAP)
    n_samples = overlay.dac0_player.shape[0] // 2
    for side, dacs, tone_mhz in ((0, DACS_0, CW_TONE_0_MHZ),
                                 (1, DACS_1, CW_TONE_1_MHZ)):
        tile = dacs[0] // PATH_PER_TILE
        tone = create_tone_samples(n_samples, DAC_SR, tone_mhz * 1e6, CW_AMP)
        write_tone_to_tile_player(overlay, tone, tile)
        print("side %d: DAC tile %d loaded with the %.6f MHz tone"
              % (side, tile, tone_mhz))
    return overlay

def capture_both_sides(overlay, gains, phases):
    """Drive the DACs with one gain and phase table and take one block, TX and RX together.

    They all start on the same trigger edge, so the phase of what comes back is a
    property of the path and not of when the trigger fired.
    """
    overlay.d_gain = gains
    overlay.d_phases = phases
    overlay.configure_dacs()
    overlay.da = 2
    raw = capture_aligned(overlay, TRIG_HOLD_S)
    iq = convert_raw_to_iq(raw, N_CH)
    if iq.shape[1] < N_CAP:
        raise ValueError("capture is %d samples long, N_CAP asks for %d"
                         % (iq.shape[1], N_CAP))
    return iq[:, :N_CAP]

OVERLAY = setup_board()
print("board ready: %d ADC channels open, both sides transmitting" % N_CH)

## 6. DSIC - digital cancellation

`DSIC_ALGORITHM` picks a canceller out of the lib. Each one is a **pair** - the
fit that learns a model from a training capture, and the call that rebuilds the
interference from it - because they do not all return the same thing:

| name | model | what it is for |
|---|---|---|
| `linear_wiener` | taps | the least squares answer in one shot, no step size |
| `linear_lms` | taps | the same answer walked to, one sample at a time |
| `non_linear_lms` | taps per odd order | when the amplifier, not just the multipath, shaped the leakage |
| `non_linear_wiener` | taps and coefficients | filter first, amplifier fitted behind it |

**One model per side**, each fitted against *its own* tone, and each fitted on a
capture taken while **the other side was muted**. That solo block holds nothing
but the side's own leakage, so the fit has no link to learn and cannot subtract
any of it back out later - which is what makes the preservation column further
down a measurement of the physics and not of the fit.

The references are written at **unit power**, not at `CW_AMP`. `DSIC_MU` is a step
against the input power, and full scale would need a step 80 dB smaller to stay
stable. The real amplitude ends up in the taps.

In [ ]:
FIT_DSIC, APPLY_DSIC = DSIC_ALGORITHMS[DSIC_ALGORITHM]

def create_tx_reference(tone_mhz):
    """One side's transmitted tone at the rate the receiver hands data back at."""
    time_axis = np.arange(N_CAP) / ADC_SR
    return np.exp(2j * np.pi * tone_mhz * 1e6 * time_axis)

TX_REFERENCE = {0: create_tx_reference(CW_TONE_0_MHZ),
                1: create_tx_reference(CW_TONE_1_MHZ)}

print("dsic ready: %s, %d tap(s), one model per side" % (DSIC_ALGORITHM, DSIC_TAPS))

## 7. Run the eight captures

| # | transmit | who is on | what the capture is for | stage it becomes |
|---|---|---|---|---|
| 1 | uniform | both | the reference every number is read against | no cancellation |
| 2 | uniform | side 0 only | train side 0's canceller | - |
| 3 | uniform | side 1 only | train side 1's canceller | - |
| 4 | uniform | both | run them | DSIC |
| 5 | BSIC | both | beamforming alone | BSIC |
| 6 | BSIC | side 0 only | train side 0's canceller again | - |
| 7 | BSIC | side 1 only | train side 1's canceller again | - |
| 8 | BSIC | both | run them | BSIC + DSIC |

A canceller **trains on one block and runs on the next**. Fitting and scoring on
the same block would make the depth a fit quality rather than a cancellation.

Each side trains alone, with the other one quiet, so a canceller only ever sees
its own leakage. The four solo captures are never scored - a block with no link in
it has no preservation to report. The cancellers are retrained after BSIC because
every beam moved, so a model fitted before describes a channel that no longer
exists.

In [ ]:
STAGE_NAMES = ["no cancellation", "DSIC", "BSIC", "BSIC + DSIC"]
SIDES = (0, 1)

def run_stages(channels):
    """The eight captures, turned into the four stages, for both sides at once."""
    uniform = create_uniform_weights(channels)
    reference = capture_stage(uniform, "1/8 reference, no bsic no dsic")
    uniform_training = capture_training(uniform, "2,3/8 dsic training, no bsic")
    dsic = capture_stage(uniform, "4/8 dsic active, no bsic")
    bsic = capture_stage(BSIC_WEIGHTS, "5/8 bsic on, no dsic")
    bsic_training = capture_training(BSIC_WEIGHTS, "6,7/8 dsic training, bsic on")
    both = capture_stage(BSIC_WEIGHTS, "8/8 dsic active, bsic on")

    run_dsic_on_stage(dsic, uniform_training)
    run_dsic_on_stage(both, bsic_training)
    return {"no cancellation": reference, "DSIC": dsic, "BSIC": bsic,
            "BSIC + DSIC": both}

def capture_stage(weights, label):
    """One aligned capture, each side's own receivers combined the way it listens.

    Summed in numpy since this board has no analog receive beamformer. The raw
    antenna rows are kept anyway - saturation is per antenna and no combiner
    undoes it.
    """
    gains, phases = create_hardware_weights(weights)
    iq = capture_both_sides(OVERLAY, gains, phases)
    stage = {}
    for side, adcs in ((0, ADCS_0), (1, ADCS_1)):
        antennas = iq[adcs]
        stream = weights["rx_%d" % side] @ antennas
        stage[side] = {"antennas": antennas,
                       "stream": stream,
                       "residual": stream,
                       "antenna_power": np.mean(np.abs(antennas) ** 2),
                       "dsic_model": None}
    print("  %-32s side 0 %.4e   side 1 %.4e"
          % (label,
             np.mean(np.abs(stage[0]["stream"]) ** 2),
             np.mean(np.abs(stage[1]["stream"]) ** 2)))
    return stage

def capture_training(weights, label):
    """One solo capture per side, the other muted - the blocks the cancellers fit on.

    Two captures and not one: a side has to be the only transmitter in the block it
    trains on, and both of them cannot be alone at the same time.
    """
    gains, phases = create_hardware_weights(weights)
    training = {}
    for side, adcs in ((0, ADCS_0), (1, ADCS_1)):
        quiet_dacs = DACS_1 if side == 0 else DACS_0
        iq = capture_both_sides(OVERLAY, create_silent_gains(gains, quiet_dacs), phases)
        training[side] = weights["rx_%d" % side] @ iq[adcs]
    print("  %-32s side 0 %.4e   side 1 %.4e"
          % (label,
             np.mean(np.abs(training[0]) ** 2),
             np.mean(np.abs(training[1]) ** 2)))
    return training

def run_dsic_on_stage(stage, training):
    """Fit each side's canceller on its solo capture and subtract it from this one."""
    for side in SIDES:
        reference = TX_REFERENCE[side]
        model = FIT_DSIC(reference[:DSIC_TRAIN_N], training[side][:DSIC_TRAIN_N],
                         DSIC_TAPS, DSIC_MU)
        stage[side]["dsic_model"] = model
        stage[side]["residual"] = stage[side]["stream"] - APPLY_DSIC(model, reference)

STAGES = run_stages(CHANNELS)
for name in STAGE_NAMES:
    print("%-16s side 0 %.4e   side 1 %.4e"
          % (name,
             np.mean(np.abs(STAGES[name][0]["residual"]) ** 2),
             np.mean(np.abs(STAGES[name][1]["residual"]) ** 2)))

## 8. Analyze the results

Every number comes off **two FFT bins** of a side's own residual stream: the bin
its own tone sits on, and the bin the other side's tone sits on.

| column | what it is | good |
|---|---|---|
| `depth_db` | own tone in capture 1 over own tone now | large |
| `preservation_db` | other side's tone now over the same tone in capture 1 | near 0 |
| `sir_db` | other side's tone over own tone, in this stage | large |
| `antenna_db` | per antenna level before combining, the saturation check | context |

`sir_db` is the one to quote. Depth on its own can be bought by simply
transmitting less, and a side that cancelled 60 dB by wrecking its own link has
not built a full duplex radio - the SIR catches that, the depth does not.

Two biases worth remembering: the gain ceiling normalises the **peak** element and
not the total, so a tapered BSIC vector transmits less power and part of its depth
is just that. And nothing is averaged - every number is one block.

In [ ]:
def measure_tone_power(signal, tone_mhz):
    """Power in the single FFT bin a tone sits on, floored so a perfect null is finite."""
    spectrum = np.fft.fft(signal)
    index = find_tone_bin(tone_mhz, len(signal), ADC_SR)
    power = abs(spectrum[index] / len(signal)) ** 2
    return max(power, 1e-30)

def analyze_results(stages):
    """Depth, preservation and SIR per side per stage, all against capture 1."""
    own_tone = {0: CW_TONE_0_MHZ, 1: CW_TONE_1_MHZ}
    other_tone = {0: CW_TONE_1_MHZ, 1: CW_TONE_0_MHZ}
    reference = stages["no cancellation"]
    rows = []
    for side in SIDES:
        reference_own = measure_tone_power(reference[side]["residual"], own_tone[side])
        reference_other = measure_tone_power(reference[side]["residual"],
                                             other_tone[side])
        for name in STAGE_NAMES:
            residual = stages[name][side]["residual"]
            own = measure_tone_power(residual, own_tone[side])
            other = measure_tone_power(residual, other_tone[side])
            rows.append({"side": side,
                         "stage": name,
                         "depth_db": 10 * np.log10(reference_own / own),
                         "preservation_db": 10 * np.log10(other / reference_other),
                         "sir_db": 10 * np.log10(other / own),
                         "antenna_db": 10 * np.log10(stages[name][side]["antenna_power"])})
    return rows

def print_result_table(rows):
    """One block per side, one row per stage."""
    headers = ["depth_db", "preservation_db", "sir_db", "antenna_db"]
    for side in SIDES:
        print("side %d" % side)
        print("%-16s" % "stage" + "".join("%17s" % header for header in headers))
        for row in rows:
            if row["side"] != side:
                continue
            print("%-16s" % row["stage"]
                  + "".join("%17.2f" % row[header] for header in headers))
        print("")

RESULTS = analyze_results(STAGES)
print_result_table(RESULTS)

## 9. Plot

The residual spectrum per side, then the two headline numbers as bars.

In the spectra the red line is the side's own tone, the one that should collapse
between stages, and the green line is the other side's tone, the one that should
not move. A stage whose red line has dropped into the noise floor has cancelled as
far as this bench can see.

In [ ]:
import matplotlib.pyplot as plt

def plot_stage_spectra(stages):
    """Each side's residual spectrum per stage, with both tone bins marked."""
    frequency_mhz = np.fft.fftshift(np.fft.fftfreq(N_CAP, 1 / ADC_SR)) / 1e6
    own_tone = {0: CW_TONE_0_MHZ, 1: CW_TONE_1_MHZ}
    other_tone = {0: CW_TONE_1_MHZ, 1: CW_TONE_0_MHZ}
    figure, axes = plt.subplots(1, 2, figsize=(13, 4))
    for side in SIDES:
        axis = axes[side]
        for name in STAGE_NAMES:
            spectrum = np.fft.fftshift(np.fft.fft(stages[name][side]["residual"]))
            power_db = 10 * np.log10(np.abs(spectrum / N_CAP) ** 2 + 1e-30)
            axis.plot(frequency_mhz, power_db, label=name, linewidth=1)
        axis.axvline(own_tone[side], color="tab:red", linestyle="--", linewidth=1)
        axis.axvline(other_tone[side], color="tab:green", linestyle="--", linewidth=1)
        axis.set_xlim(min(own_tone[side], other_tone[side]) - 1,
                      max(own_tone[side], other_tone[side]) + 1)
        axis.set_xlabel("baseband frequency [MHz]")
        axis.set_ylabel("power [dB]")
        axis.set_title("side %d - red: its own tone, green: the link" % side)
        axis.grid(alpha=0.3)
        axis.legend()
    figure.tight_layout()

def plot_stage_summary(rows):
    """Depth and preservation per stage, the two sides beside each other."""
    positions = np.arange(len(STAGE_NAMES))
    width = 0.35
    panels = [("cancellation depth [dB]", "depth_db"),
              ("link preservation [dB]", "preservation_db")]
    figure, axes = plt.subplots(1, len(panels), figsize=(13, 4))
    for axis, (title, key) in zip(axes, panels):
        for side in SIDES:
            values = []
            for name in STAGE_NAMES:
                for row in rows:
                    if row["side"] == side and row["stage"] == name:
                        values.append(row[key])
            offset = width * (side - 0.5)
            bars = axis.bar(positions + offset, values, width, label="side %d" % side)
            for position, value in zip(positions + offset, values):
                axis.text(position, value, "%.1f" % value, ha="center", fontsize=8,
                          va="bottom" if value >= 0 else "top")
        axis.set_xticks(positions)
        axis.set_xticklabels(STAGE_NAMES, rotation=20)
        axis.set_title(title)
        axis.grid(axis="y", alpha=0.3)
        axis.legend()
    figure.tight_layout()

plot_stage_spectra(STAGES)
plot_stage_summary(RESULTS)
plt.show()

## 10. Save

| file | what it holds |
|---|---|
| `params.json` | every setting the run used, both tones included |
| `results.md` | the two tables, both sides' weights and both DSIC models |
| `tx_signal.npz` | the two transmitted waveforms, at the receive rate |
| `rx_signal.npz` | capture 1, one row per antenna, per side |
| `clean_rx_signal.npz` | capture 5 combined and cancelled, one stream per side |

The folder is emptied first, so nothing stale reads as a result of this run. Only
the files sitting directly in it.

In [ ]:
def save_sic_run(stages, weights, rows):
    """Replace the folder with the settings, the report and both sides' signals."""
    clear_dir(SIC_DIR)
    save_sic_params(SIC_DIR)
    save_results_report(SIC_DIR, rows, stages, weights)
    save_signals(SIC_DIR, stages)
    print("saved %s: %s" % (SIC_DIR, sorted(os.listdir(SIC_DIR))))

def save_sic_params(folder):
    """Record the settings the numbers came from, so a result can be traced back."""
    params = {"dacs_0": DACS_0, "adcs_0": ADCS_0,
              "dacs_1": DACS_1, "adcs_1": ADCS_1,
              "channels_dir": CHANNELS_DIR,
              "bsic_algorithm": BSIC_ALGORITHM,
              "dsic_algorithm": DSIC_ALGORITHM,
              "dac_nco": DAC_NCO, "dac_zone": DAC_ZONE,
              "adc_nco": ADC_NCO, "adc_zone": ADC_ZONE,
              "cw_tone_0_mhz": CW_TONE_0_MHZ, "cw_tone_1_mhz": CW_TONE_1_MHZ,
              "cw_amp": CW_AMP,
              "snr_db": SNR_DB,
              "bsic_max_gain": BSIC_MAX_GAIN,
              "max_phase_deg": MAX_PHASE_DEG,
              "dsic_taps": DSIC_TAPS, "dsic_mu": DSIC_MU,
              "dsic_train_n": DSIC_TRAIN_N,
              "n_cap": N_CAP, "dac_sr": DAC_SR, "adc_sr": ADC_SR}
    save_json_params(folder, params)

def save_results_report(folder, rows, stages, weights):
    """The tables, both sides' weights and both DSIC models, as one markdown file."""
    headers = ["depth_db", "preservation_db", "sir_db", "antenna_db"]
    lines = ["# Two sided full duplex SIC run", "",
             "BSIC: `%s` - DSIC: `%s`" % (BSIC_ALGORITHM, DSIC_ALGORITHM), "",
             "Side 0 transmits %.6f MHz, side 1 transmits %.6f MHz."
             % (CW_TONE_0_MHZ, CW_TONE_1_MHZ), ""]
    for side in SIDES:
        dacs = DACS_0 if side == 0 else DACS_1
        adcs = ADCS_0 if side == 0 else ADCS_1
        lines.append("## Side %d" % side)
        lines.append("")
        lines.append("| stage | " + " | ".join(headers) + " |")
        lines.append("|---" * (len(headers) + 1) + "|")
        for row in rows:
            if row["side"] != side:
                continue
            values = " | ".join("%.2f" % row[header] for header in headers)
            lines.append("| %s | %s |" % (row["stage"], values))
        lines.append("")
        lines.append("Transmit, per DAC:")
        lines.append("")
        lines.append("| DAC | gain | phase [deg] |")
        lines.append("|---|---|---|")
        for dac in dacs:
            lines.append("| %d | %.4f | %+.3f |"
                         % (dac, BSIC_GAINS[dac], BSIC_PHASES[dac]))
        lines.append("")
        lines.append("Receive combine, per ADC:")
        lines.append("")
        lines.append("| ADC | magnitude | phase [deg] |")
        lines.append("|---|---|---|")
        for adc, value in zip(adcs, weights["rx_%d" % side]):
            lines.append("| %d | %.4f | %+.3f |"
                         % (adc, abs(value), np.degrees(np.angle(value))))
        lines.append("")
        lines.append("DSIC model after BSIC:")
        lines.append("")
        lines.append("| block | index | magnitude | phase [deg] |")
        lines.append("|---|---|---|---|")
        model = stages["BSIC + DSIC"][side]["dsic_model"]
        if isinstance(model, tuple):
            blocks = [("taps", model[0]), ("coeffs", model[1])]
        else:
            blocks = [("taps", model)]
        for name, values in blocks:
            for index, value in enumerate(values):
                lines.append("| %s | %d | %.6g | %+.3f |"
                             % (name, index, abs(value), np.degrees(np.angle(value))))
        lines.append("")
    with open(os.path.join(folder, "results.md"), "w") as handle:
        handle.write("\n".join(lines) + "\n")

def save_signals(folder, stages):
    """What each side sent, what its antennas heard, and what it was left with."""
    np.savez(os.path.join(folder, "tx_signal.npz"),
             tx_signal_0=TX_REFERENCE[0], tx_signal_1=TX_REFERENCE[1])
    np.savez(os.path.join(folder, "rx_signal.npz"),
             rx_signal_0=stages["no cancellation"][0]["antennas"],
             rx_signal_1=stages["no cancellation"][1]["antennas"])
    np.savez(os.path.join(folder, "clean_rx_signal.npz"),
             clean_rx_signal_0=stages["BSIC + DSIC"][0]["residual"],
             clean_rx_signal_1=stages["BSIC + DSIC"][1]["residual"])

save_sic_run(STAGES, BSIC_WEIGHTS, RESULTS)

## 11. Stop

Switch both transmitters off when you are done.

In [ ]:
OVERLAY.dacs_off()
print("dacs off")